In [3]:
# ==========================================
# 0. IMPORTS
# ==========================================
import torch
import sentencepiece
from huggingface_hub import hf_hub_download
import moshi.utils.compile
from moshi.models import loaders, LMGen

# ==========================================
# 1. THE DYNAMIC GRAPH KILLER
# ==========================================
print("🔍 Hunting for Kyutai's CUDA Graph compiler...")
patched_any = False
for name, obj in vars(moshi.utils.compile).items():
    if isinstance(obj, type) and hasattr(obj, '__call__'):   # FIX 1: was '_call_'
        def make_bypass(orig_call):
            def bypass_call(self, *args, **kwargs):
                if hasattr(self, 'func'):
                    return self.func(*args, **kwargs)
                return orig_call(self, *args, **kwargs)
            return bypass_call
        obj.__call__ = make_bypass(obj.__call__)
        print(f"✅ Successfully neutralized CUDA Graphs in: moshi.utils.compile.{name}")
        patched_any = True

if not patched_any:
    print("⚠️  No CUDAGraphed classes found — patch may be unnecessary for this version.")

# ==========================================
# 2. PATHS & DEVICE CONFIG
# ==========================================
DEVICE_HOME = "cuda:0"
DEVICE_WORK = "cuda:1"

# ==========================================
# 3. LOAD MODELS & SHARD ACROSS GPUs
# ==========================================
repo_id = "kyutai/moshika-pytorch-bf16"

print("\nDownloading Mimi weights...")
mimi = loaders.get_mimi(
    hf_hub_download(repo_id, "tokenizer-e351c8d8-checkpoint125.safetensors"),
    device="cpu"
)
mimi = mimi.to(DEVICE_HOME)
mimi.set_num_codebooks(8)   # FIX 2: was set_num_codebooks(😎

print("Downloading Moshi weights (~15.4 GB)...")
moshi_lm = loaders.get_moshi_lm(   # renamed to avoid shadowing the `moshi` package
    hf_hub_download(repo_id, "model.safetensors"),
    device="cpu"
)
text_tokenizer = sentencepiece.SentencePieceProcessor(
    hf_hub_download(repo_id, "tokenizer_spm_32k_3.model")
)

# ==========================================
# 4. SHARD TRANSFORMER LAYERS
# ==========================================
print("\n🛠️ Sharding Model across GPUs...")

# FIX 3: hook always receives args as a tuple — simplified and corrected
def shard_hook(module, args):
    try:
        target_dev = next(module.parameters()).device
    except StopIteration:
        try:
            target_dev = next(module.buffers()).device
        except StopIteration:
            return args
    return tuple(
        a.to(target_dev) if isinstance(a, torch.Tensor) else a
        for a in args
    )

layers = moshi_lm.transformer.layers
mid = len(layers) // 2

# HOME: embeddings + first half of layers
moshi_lm.emb.to(DEVICE_HOME)
moshi_lm.text_emb.to(DEVICE_HOME)
for i in range(mid):
    layers[i].to(DEVICE_HOME)
    layers[i].register_forward_pre_hook(shard_hook)

# WORK: second half of layers
for i in range(mid, len(layers)):
    layers[i].to(DEVICE_WORK)
    layers[i].register_forward_pre_hook(shard_hook)

# Norm stays on HOME (produces output consumed by HOME-side heads)
if hasattr(moshi_lm.transformer, "norm"):
    moshi_lm.transformer.norm.to(DEVICE_HOME)
    moshi_lm.transformer.norm.register_forward_pre_hook(shard_hook)

# All other top-level children (heads, depformer, linears, etc.) → HOME
for name, module in moshi_lm.named_children():
    if name not in ["emb", "text_emb", "transformer"]:
        module.to(DEVICE_HOME)
        module.register_forward_pre_hook(shard_hook)

# FIX 4: move scalar/tensor attributes AFTER layer placement is done
for attr in ['initial', 'delays', 'zero_token_id']:
    if hasattr(moshi_lm, attr):
        t = getattr(moshi_lm, attr)
        if isinstance(t, torch.Tensor):
            setattr(moshi_lm, attr, t.to(DEVICE_HOME))

# FIX 5: only re-home buffers that still belong to the top-level module
# (layer buffers are already on their correct device after the .to() calls above)
for name, buf in moshi_lm.named_buffers(recurse=False):
    buf.data = buf.data.to(DEVICE_HOME)

# ==========================================
# 5. CONSTRUCT LMGen
# ==========================================
lm_gen = LMGen(moshi_lm, temp=0.8, temp_text=0.7)
print("\n✅ LMGen ready.")

🔍 Hunting for Kyutai's CUDA Graph compiler...
✅ Successfully neutralized CUDA Graphs in: moshi.utils.compile.Checkpoint
✅ Successfully neutralized CUDA Graphs in: moshi.utils.compile.CUDAGraphed



tokenizer-e351c8d8-checkpoint125.safeten(…):   0%|          | 0.00/385M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/15.4G [00:00<?, ?B/s]

tokenizer_spm_32k_3.model:   0%|          | 0.00/553k [00:00<?, ?B/s]


🛠️ Sharding Model across GPUs...

✅ LMGen ready.


In [1]:
!pip install moshi speechbrain sentencepiece

INFO: pip is looking at multiple versions of torchaudio to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 39.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 74.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.7/124.7 MB 2.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 10.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 59.9 MB/s eta 0:00:00

In [4]:
# ==========================================
# INFERENCE TEST — GPU BOUNDARY VALIDATION
# ==========================================
import torch
import numpy as np

print("=" * 55)
print("INFERENCE TEST: Sharded LMGen across cuda:0 / cuda:1")
print("=" * 55)

# ── 1. Sanity-check device placement ───────────────────────
mid = len(moshi_lm.transformer.layers) // 2
home_dev = next(moshi_lm.transformer.layers[0].parameters()).device
work_dev = next(moshi_lm.transformer.layers[mid].parameters()).device
print(f"\n[Device check]")
print(f"  Layer 0   → {home_dev}   (expected cuda:0)")
print(f"  Layer {mid} → {work_dev}  (expected cuda:1)")
print(f"  Emb       → {next(moshi_lm.emb.parameters()).device}")
assert str(home_dev) == "cuda:0", "Layer 0 not on cuda:0!"
assert str(work_dev) == "cuda:1", f"Layer {mid} not on cuda:1!"
print("  ✅ Device placement confirmed.\n")

# ── 2. Craft a silent dummy audio input ────────────────────
# Mimi requires multiples of 1920 samples; 1920 × 5 = 9600 ≈ 0.6 s @ 24 kHz
N_CHUNKS   = 5          # number of 1920-sample frames
CHUNK_SIZE = 1920
SAMPLE_RATE = 24000

silent_audio = torch.zeros(1, 1, N_CHUNKS * CHUNK_SIZE)   # [B, C, T]
silent_audio = silent_audio.to(DEVICE_HOME)

print(f"[Input] Silent audio: {silent_audio.shape}  ({N_CHUNKS * CHUNK_SIZE / SAMPLE_RATE:.2f}s)")

# ── 3. Encode audio → RVQ tokens via Mimi ──────────────────
mimi.eval()
with torch.no_grad():
    codes = mimi.encode(silent_audio)          # [1, 8, T_frames]

print(f"[Mimi] Encoded tokens: {codes.shape}  (8 codebooks × {codes.shape[-1]} frames)")

# Moshi expects 17-codebook input: row 0 = text placeholder, rows 1-8 = Mimi acoustic
T = codes.shape[-1]
text_placeholder = torch.full((1, 1, T), fill_value=moshi_lm.zero_token_id,
                               dtype=torch.long, device=DEVICE_HOME)
# pad remaining codebooks 9-16 with zero_token_id
acoustic_pad     = torch.full((1, 8, T), fill_value=moshi_lm.zero_token_id,
                               dtype=torch.long, device=DEVICE_HOME)
full_input = torch.cat([text_placeholder, codes.to(DEVICE_HOME), acoustic_pad], dim=1)
# shape: [1, 17, T]
print(f"[Input tokens] Full token tensor: {full_input.shape}  (17 × {T})")

# ── 4. Streaming decode through LMGen ──────────────────────
print(f"\n[LMGen] Running {T} streaming steps...")
moshi_lm.eval()

generated_text_tokens  = []
generated_audio_tokens = []   # codebook 0 of the *output* acoustic stream

step_errors = []
with torch.no_grad():
    lm_gen.reset_streaming()
    for t in range(T):
        token_slice = full_input[:, :, t : t + 1]   # [1, 17, 1]
        try:
            out = lm_gen.step(token_slice)           # [1, 17, 1]
            # out[:, 0, :] = text logit/token
            # out[:, 1:,  :] = acoustic tokens
            generated_text_tokens.append(out[:, 0, 0].item())
            generated_audio_tokens.append(out[:, 1, 0].item())
        except Exception as e:
            step_errors.append((t, str(e)))
            if len(step_errors) >= 3:
                print(f"  ⛔ Too many step errors — aborting loop.")
                break

print(f"  Steps completed: {T - len(step_errors)} / {T}")
if step_errors:
    for t, err in step_errors:
        print(f"  ⚠️  Step {t}: {err}")

# ── 5. Cross-device tensor flow check ──────────────────────
print("\n[GPU Memory after inference]")
for i in range(torch.cuda.device_count()):
    alloc  = torch.cuda.memory_allocated(i)  / 1e9
    reserv = torch.cuda.memory_reserved(i)   / 1e9
    print(f"  cuda:{i}  allocated={alloc:.2f} GB  reserved={reserv:.2f} GB")

# ── 6. Decode text tokens → readable string ────────────────
NON_WORD = moshi_lm.zero_token_id
real_text = [tok for tok in generated_text_tokens if tok != NON_WORD]
if real_text:
    try:
        decoded = text_tokenizer.decode(real_text)
        print(f"\n[Text output]  \"{decoded}\"")
    except Exception as e:
        print(f"\n[Text output]  decode failed: {e}")
        print(f"  Raw tokens: {real_text[:20]}")
else:
    print(f"\n[Text output]  All zero-token (expected for silent input — model has nothing to transcribe)")

# ── 7. Audio token sanity ───────────────────────────────────
print(f"\n[Audio tokens (codebook 0, first 10 steps)]: {generated_audio_tokens[:10]}")
unique_audio = len(set(generated_audio_tokens))
print(f"  Unique values across {T} steps: {unique_audio}  ", end="")
if unique_audio > 1:
    print("✅  (model is generating non-constant output)")
else:
    print("⚠️  (all identical — model may be collapsed; check temp settings)")

# ── 8. Final verdict ────────────────────────────────────────
print("\n" + "=" * 55)
passed = (len(step_errors) == 0)
if passed:
    print("✅  PASS — All steps completed, tensors crossed the GPU")
    print("         boundary without errors. Safe to proceed to KD.")
else:
    print("❌  FAIL — See step errors above.")
print("=" * 55)

INFERENCE TEST: Sharded LMGen across cuda:0 / cuda:1

[Device check]
  Layer 0   → cuda:0   (expected cuda:0)
  Layer 16 → cuda:1  (expected cuda:1)
  Emb       → cuda:0
  ✅ Device placement confirmed.

[Input] Silent audio: torch.Size([1, 1, 9600])  (0.40s)
[Mimi] Encoded tokens: torch.Size([1, 8, 5])  (8 codebooks × 5 frames)
[Input tokens] Full token tensor: torch.Size([1, 17, 5])  (17 × 5)

[LMGen] Running 5 streaming steps...


AssertionError: 

In [2]:
# GPU 0 — teacher (fp16 cast from bf16)
from transformers import AutoModelForCausalLM
teacher = AutoModelForCausalLM.from_pretrained(
    "kyutai/moshiko-pytorch-bf16", torch_dtype=torch.float16
).to("cuda:0")

# GPU 1 — student
student_backbone = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM2-1.7B", torch_dtype=torch.float16
).to("cuda:1")

ImportError: cannot import name 'is_offline_mode' from 'huggingface_hub' (/usr/local/lib/python3.12/dist-packages/huggingface_hub/__init__.py)